Silver Layer : Transforming all the datasets and fix all the datatypes mismatch and droping nulls

First creating the silver schema

In [0]:
%sql
create database if not exists cash_flow_project.cash_flow_silver

1. checking_accounts_main

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.table("cash_flow_project.cash_flow_bronze.checking_accounts_main")
df_silver = (df
    .withColumn("date",  F.to_date(F.col("date"), "yyyy-MM-dd"))
    .withColumn("day",   F.dayofmonth("date"))
    .withColumn("month", F.month("date"))
    .withColumn("year",  F.year("date"))
    # signed amount: Debit = negative, Credit = positive
    .withColumn("signed_amount",
        F.when(F.col("type") == "Debit",   F.col("amount") * -1)
         .otherwise(F.col("amount")))
    .dropDuplicates(["transaction_id"])
)

In [0]:
(df_silver.write.format("delta").mode("overwrite")
 .saveAsTable("cash_flow_project.cash_flow_silver.checking_accounts_main"))
display(df_silver.limit(5))

2. checking_account_secondary

In [0]:
df = spark.table("cash_flow_project.cash_flow_bronze.checking_account_secondary")
df_silver = (df
    .withColumn("date",  F.to_date(F.col("date"), "yyyy-MM-dd"))
    .withColumn("day",   F.dayofmonth("date"))
    .withColumn("month", F.month("date"))
    .withColumn("year",  F.year("date"))
    .withColumn("signed_amount",
        F.when(F.col("type") == "Debit",   F.col("amount") * -1)
         .otherwise(F.col("amount")))
    .dropDuplicates(["transaction_id"])
)

In [0]:
(df_silver.write.format("delta").mode("overwrite")
 .saveAsTable("cash_flow_project.cash_flow_silver.checking_account_secondary"))
display(df_silver.limit(5))

3. coffee_shop_sales

In [0]:
df = spark.table("cash_flow_project.cash_flow_bronze.coffee_shop_sales")
df_silver = (df
    .withColumn("transaction_date", F.to_date(F.col("transaction_date"), "yyyy-MM-dd"))
    .withColumn("year",  F.year("transaction_date"))
    .withColumn("month", F.month("transaction_date"))
    .withColumn("day",   F.dayofmonth("transaction_date"))
    .withColumn("hour",  F.hour("transaction_time"))
    .withColumn("revenue", F.round(F.col("transaction_qty") * F.col("unit_price"), 2))
    .withColumn("time_full", F.date_format("transaction_time", "HH:mm:ss"))
    .dropDuplicates(["transaction_id"])
)

In [0]:
(df_silver.write.format("delta").mode("overwrite")
 .saveAsTable("cash_flow_project.cash_flow_silver.coffee_shop_sales"))
display(df_silver.limit(5))

4. credit_card_account

In [0]:
df = spark.table("cash_flow_project.cash_flow_bronze.credit_card_account")
df_silver = (df
    .withColumn("date",  F.to_date(F.col("date"), "yyyy-MM-dd"))
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day",   F.dayofmonth("date"))
    .withColumn("signed_amount",
        F.when(F.col("type") == "Debit",   F.col("amount") * -1)
         .otherwise(F.col("amount")))
    .dropDuplicates(["transaction_id"])
)

In [0]:
(df_silver.write.format("delta").mode("overwrite")
 .saveAsTable("cash_flow_project.cash_flow_silver.credit_card_account"))
display(df_silver.limit(5))

5. gusto_payroll

In [0]:
df = spark.table("cash_flow_project.cash_flow_bronze.gusto_payroll")
df_silver = (df
    .withColumn("pay_date", F.to_date(F.col("pay_date"), "yyyy-MM-dd"))
    .withColumn("year",  F.year("pay_date"))
    .withColumn("month", F.month("pay_date"))
    .withColumn("day",   F.dayofmonth("pay_date"))
    .dropDuplicates(["employee_id", "pay_date"])  
)

In [0]:
(df_silver.write.format("delta").mode("overwrite")
 .saveAsTable("cash_flow_project.cash_flow_silver.gusto_payroll"))
display(df_silver.limit(5))